# seccal: fine-tune `unsloth/Qwen3-4B-Instruct-2507` (preset `qwen3-4b`)

**Before running:** `Runtime → Change runtime type → T4 GPU` (free tier is enough).

Then `Runtime → Run all`. Embedded: training script + 399 train / 52 val rows, with a SHA-256 manifest that is checked before anything is extracted. Nothing else to upload. Output GGUFs are copied to your Google Drive at `MyDrive/seccal-12b/` (one Drive authorization prompt). Roughly 10–25 minutes on a T4 including installs.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

## 1. Verify and unpack the embedded training script and data

`EXPECTED` is the SHA-256 of each bundled file as generated from the repository's tracked sources. If any member differs, the cell stops before extracting anything.

In [ ]:
import base64, io, zipfile, os, hashlib
EXPECTED = {
 "train/sft_unsloth.py": "92d7c3b12e68f7ac7bbda4f905cd0e5b3b20baff90e0b39a97dc02f08b44c098",
 "data/sft/train.jsonl": "f6c84acea1c69708068e3f5af4ff1b61df2c8dd55427560fecd3279973a5747d",
 "data/sft/val.jsonl": "f85e78172ae0c743c457c858aa6f9e014e978b2870ab7b9e568396645a09dd30"
}
BUNDLE_B64 = '''...'''
z = zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64)))
assert set(z.namelist()) == set(EXPECTED), f'unexpected bundle members: {z.namelist()}'
for n in z.namelist():
    h = hashlib.sha256(z.read(n)).hexdigest()
    assert h == EXPECTED[n], f'bundle member {n} does not match the manifest ({h} != {EXPECTED[n]})'
z.extractall('/content/seccal')
os.chdir('/content/seccal')
for n in z.namelist(): print('verified', n, os.path.getsize(n), 'bytes')

## 2. Install the training stack (a few minutes)

In [ ]:
%pip install -q -U unsloth "trl>=0.15" datasets transformers
import unsloth, trl, transformers
print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| transformers', transformers.__version__)

## 3. (Optional) Hugging Face token

Only needed if the base model download asks for one (none of the presets are gated). Add a Colab secret named `HF_TOKEN` and this cell will pick it up; otherwise it is skipped.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets')
except Exception:
    print('no HF_TOKEN secret; continuing without one')

## 4. Train (`--model qwen3-4b`), then export Q4_K_M and Q8_0 GGUFs

If a 16-bit LoRA runs out of memory on the T4 (possible on 4B presets), add `--load-in-4bit`.

In [ ]:
!python train/sft_unsloth.py --model qwen3-4b --train data/sft/train.jsonl --val data/sft/val.jsonl --out out/seccal-qwen3-4b --epochs 3
!find out/seccal-qwen3-4b -name '*.gguf' -exec ls -la {} \;

## 5. Save the GGUFs to Google Drive

Approve the Drive prompt. Files land in `MyDrive/seccal-12b/`. Then on your Mac, copy them into the repo's `models/` folder as `models/seccal_qwen3-4b_Q4_K_M.gguf` and `models/seccal_qwen3-4b_Q8_0.gguf`.

In [ ]:
from google.colab import drive
import shutil, glob, os
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/seccal-12b'
os.makedirs(dst, exist_ok=True)
ggufs = glob.glob('out/seccal-qwen3-4b/**/*.gguf', recursive=True)
assert ggufs, 'no .gguf files found under out/seccal-qwen3-4b; check the training cell output'
for f in ggufs:
    shutil.copy(f, dst)
    print('copied', os.path.basename(f), round(os.path.getsize(f)/1e6), 'MB')
shutil.copy('out/seccal-qwen3-4b/train_stats.json', dst)
print('done ->', dst)

### Fallback: direct browser download

In [ ]:
from google.colab import files
import glob
ggufs = glob.glob('out/seccal-qwen3-4b/**/*.gguf', recursive=True)
assert ggufs, 'no .gguf files found under out/seccal-qwen3-4b'
for f in ggufs:
    print(f); files.download(f)